In [1]:
# Imports
import random
import time

In [2]:
def cross(A, B):
    "Cross product of elements in A and elements in B."
    return [a+b for a in A for b in B]

In [3]:
# Constants
DIGITS  = '123456789'
ROWS    = 'ABCDEFGHI'
COLS    = DIGITS
SQUARES = cross(ROWS, COLS)

UNITLIST = ([cross(ROWS, c) for c in COLS] +
            [cross(r, COLS) for r in ROWS] +
            [cross(rs, cs) for rs in ('ABC', 'DEF', 'GHI') for cs in ('123', '456', '789')])
UNITS    = dict((s, [u for u in UNITLIST if s in u])
             for s in SQUARES)
PEERS    = dict((s, set(sum(UNITS[s], []))-set([s]))
             for s in SQUARES)

# Unit Tests

In [4]:
def test():
    "A set of tests that must pass."
    assert len(SQUARES) == 81
    assert len(UNITLIST) == 27
    assert all(len(UNITS[s]) == 3 for s in SQUARES)
    assert all(len(PEERS[s]) == 20 for s in SQUARES)
    assert UNITS['C2'] == [['A2', 'B2', 'C2', 'D2', 'E2', 'F2', 'G2', 'H2', 'I2'],
                           ['C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9'],
                           ['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'C1', 'C2', 'C3']]
    assert PEERS['C2'] == set(['A2', 'B2', 'D2', 'E2', 'F2', 'G2', 'H2', 'I2',
                               'C1', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9',
                               'A1', 'A3', 'B1', 'B3'])
    print('All tests pass.')

# Parse a Grid

In [5]:

def parse_grid(grid):
    """Convert grid to a dict of possible values, {square: digits}, or
    return False if a contradiction is detected."""
    # To start, every square can be any digit; then assign values from the grid.
    values = dict((s, DIGITS) for s in SQUARES)
    for s, d in grid_values(grid).items():
        if d in DIGITS and not assign(values, s, d):
            return False  # (Fail if we can't assign d to square s.)
    return values


def grid_values(grid):
    "Convert grid into a dict of {square: char} with '0' or '.' for empties."
    chars = [c for c in grid if c in DIGITS or c in '0.']
    assert len(chars) == 81
    return dict(zip(SQUARES, chars))

# Constraint Propagation

In [6]:
def assign(values, s, d):
    """Eliminate all the other values (except d) from values[s] and propagate.
    Return values, except return False if a contradiction is detected."""
    other_values = values[s].replace(d, '')
    if all(eliminate(values, s, d2) for d2 in other_values):
        return values
    else:
        return False


def eliminate(values, s, d):
    """Eliminate d from values[s]; propagate when values or places <= 2.
    Return values, except return False if a contradiction is detected."""
    if d not in values[s]:
        return values  # Already eliminated
    values[s] = values[s].replace(d, '')
    # (1) If a square s is reduced to one value d2, then eliminate d2 from the peers.
    if len(values[s]) == 0:
        return False  # Contradiction: removed last value
    elif len(values[s]) == 1:
        d2 = values[s]
        if not all(eliminate(values, s2, d2) for s2 in PEERS[s]):
            return False
    # (2) If a unit u is reduced to only one place for a value d, then put it there.
    for u in UNITS[s]:
        dplaces = [s for s in u if d in values[s]]
        if len(dplaces) == 0:
            return False  # Contradiction: no place for this value
        elif len(dplaces) == 1:
            # d can only be in one place in unit; assign it there
            if not assign(values, dplaces[0], d):
                return False
    return values

# Display as 2-D grid

In [7]:
def display(values):
    "Display these values as a 2-D grid."
    width = 1+max(len(values[s]) for s in SQUARES)
    line = '+'.join(['-'*(width*3)]*3)
    for r in ROWS:
        # Modified display function as it displayed
        # a generator object and not a string
        print(''.join(list(''.join(values[r+c].center(width)+('|' if c in '36' else ''))
                           for c in COLS)))
        if r in 'CF':
            print(line)

# Search

In [8]:
def solve(grid): return search(parse_grid(grid))
def solve_depth_first(grid): return search_depth_first(parse_grid(grid))



def search(values):
    "Using depth-first search and propagation, try all possible values."
    if values is False:
        return False ## Failed earlier
    if all(len(values[s]) == 1 for s in SQUARES): 
        return values ## Solved!
    ## Chose the unfilled square s with the fewest possibilities
    n,s = min((len(values[s]), s) for s in SQUARES if len(values[s]) > 1)
    return some(search(assign(values.copy(), s, d)) 
		for d in values[s])

def search_depth_first(values):
    "Using depth-first search and propagation, try all possible values."
    if values is False:
        return False  # Failed earlier
    if all(len(values[s]) == 1 for s in SQUARES):
        return values  # Solved!
    # Question 2: L'implementation avec Random choice.
    s = SQUARES[random.randint(0, len(SQUARES)-1)]
    return some(search(assign(values.copy(), s, d))
                for d in values[s])



# Utilities

In [9]:
def some(seq):
    "Return some element of seq that is true."
    for e in seq:
        if e:
            return e
    return False


def from_file(filename, sep='\n'):
    "Parse a file into a list of strings, separated by sep."
    return open(filename).read().strip().split(sep)


def shuffled(seq):
    "Return a randomly shuffled copy of the input sequence."
    seq = list(seq)
    random.shuffle(seq)
    return seq

# System Test

In [10]:
def solve_all(grids, name='', showif=0.0):
    """Attempt to solve a sequence of grids. Report results.
    When showif is a number of seconds, display puzzles that take longer.
    When showif is None, don't display any puzzles."""
    def time_solve(grid):
        start = time.process_time()
        values = solve(grid)
        t = time.process_time()-start
        # Display puzzles that take long enough
        if showif is not None and t > showif:
            display(grid_values(grid))
            if values:
                display(values)
            print('(%.2f seconds)\n' % t)
        return (t, solved(values))
    times, results = zip(*[time_solve(grid) for grid in grids])
    N = len(grids)
    if N > 1:
        print("Solved %d of %d %s puzzles (avg %.2f secs (%d Hz), max %.2f secs)." % (
            sum(results), N, name, sum(times)/N, N/sum(times), max(times)))


def solved(values):
    "A puzzle is solved if each unit is a permutation of the digits 1 to 9."
    def unitsolved(unit): return set(values[s] for s in unit) == set(DIGITS)
    return values is not False and all(unitsolved(unit) for unit in UNITLIST)


def random_puzzle(N=17):
    """Make a random puzzle with N or more assignments. Restart on contradictions.
    Note the resulting puzzle is not guaranteed to be solvable, but empirically
    about 99.8% of them are solvable. Some have multiple solutions."""
    values = dict((s, DIGITS) for s in SQUARES)
    for s in shuffled(SQUARES):
        if not assign(values, s, random.choice(values[s])):
            break
        ds = [values[s] for s in SQUARES if len(values[s]) == 1]
        if len(ds) >= N and len(set(ds)) >= 8:
            return ''.join(values[s] if len(values[s]) == 1 else '.' for s in SQUARES)
    return random_puzzle(N)  # Give up and make a new puzzle

# Main

In [11]:
grid1 = '003020600900305001001806400008102900700000008006708200002609500800203009005010300'
grid2 = '4.....8.5.3..........7......2.....6.....8.4......1.......6.3.7.5..2.....1.4......'
hard1 = '.....6....59.....82....8....45........3........6..3.54...325..6..................'

if __name__ == '__main__':
    test()
    # solve_all(from_file("top95.txt"), "95sudoku", None)
    # solve_all(from_file("easy50.txt", '========'), "easy", None)
    # solve_all(from_file("easy50.txt", '========'), "easy", None)
    # solve_all(from_file("top95.txt"), "hard", None)
    # solve_all(from_file("hardest.txt"), "hardest", None)
    # solve_all([random_puzzle() for _ in range(99)], "random", 100.0)

    #############################################################################
    # Question 1
    solve_all(from_file("100sudoku.txt"), "100sudoku", None)
    #############################################################################

    #############################################################################    
    # Question 2
    #############################################################################
    print("\nComparing real depth-first search with Norvig's added criteria:")
    print("----------------------------------------")
    print("Solving \"hard1\"")
    display(grid_values(hard1))

    print("\nUsing real depth-first search")
    start = time.process_time()
    values = solve_depth_first(hard1)
    end = time.process_time()-start
    display(values)
    print(f"Solved: {solved(values)}")
    print("Process took %.0f ms" % (end * 1000))

    print("\nUsing depth first search with Norvig's added criteria")
    start = time.process_time()
    values = solve(hard1)
    end = time.process_time()-start
    display(values)
    print(f"Solved: {solved(values)}")
    print("Process took %.0f ms" % (end * 1000))



All tests pass.
Solved 100 of 100 100sudoku puzzles (avg 0.00 secs (376 Hz), max 0.02 secs).

Comparing real depth-first search with Norvig's added criteria:
----------------------------------------
Solving "hard1"
. . . |. . 6 |. . . 
. 5 9 |. . . |. . 8 
2 . . |. . 8 |. . . 
------+------+------
. 4 5 |. . . |. . . 
. . 3 |. . . |. . . 
. . 6 |. . 3 |. 5 4 
------+------+------
. . . |3 2 5 |. . 6 
. . . |. . . |. . . 
. . . |. . . |. . . 

Using real depth-first search
4 3 8 |9 5 6 |2 7 1 
6 5 9 |2 1 7 |3 4 8 
2 7 1 |4 3 8 |5 6 9 
------+------+------
1 4 5 |6 8 2 |9 3 7 
7 8 3 |5 4 9 |6 1 2 
9 2 6 |1 7 3 |8 5 4 
------+------+------
8 1 4 |3 2 5 |7 9 6 
5 6 2 |7 9 1 |4 8 3 
3 9 7 |8 6 4 |1 2 5 
Solved: True
Process took 0 ms

Using depth first search with Norvig's added criteria
4 3 8 |7 9 6 |2 1 5 
6 5 9 |1 3 2 |4 7 8 
2 7 1 |4 5 8 |6 9 3 
------+------+------
8 4 5 |2 1 9 |3 6 7 
7 1 3 |5 6 4 |8 2 9 
9 2 6 |8 7 3 |1 5 4 
------+------+------
1 9 4 |3 2 5 |7 8 6 
3 6 2 |9 8 7 |5 4